In [1]:
#| default_exp support

In [2]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Probe and install kernel support into a project's Python environment.

Every question here is answered by starting the interpreter being asked about. An interpreter that
is missing, broken or too slow reports the failure in its result. Only the two installers raise.

In [3]:
#| export
"Probe and install kernel support into a project's Python environment."

"Probe and install kernel support into a project's Python environment."

In [4]:
#| export
from __future__ import annotations

In [5]:
#| export
import os, shutil, subprocess, sys, threading, time

In [6]:
#| export
from pathlib import Path

In [7]:
#| export
#: This interpreter's minor version. A kernel may borrow this `sys.path` only when it matches.
HOST_PY = tuple(sys.version_info[:2])

In [8]:
#| export
#: What a kernel that cannot borrow Leela's own copy has to have of its own. `dhrishti` and not
#: its imports: numpy and pandas are the two that are noticed missing, but IPython comes with it
#: and the bundle's is bytecode for another version. Unpinned, because the versions are the
#: project's choice. Pinned by `tests/test_runtime.py`.
INSPECTOR = ('dhrishti',)

In [9]:
#| export
#: Seconds an inspector answer is reused for. A person who installs pandas by hand should not have
#: to restart Leela to be believed.
INSPECTOR_TTL = 90

In [10]:
#| export
#: What a project environment needs to run either kernel. `ipymini` names `comm`, `ipython` and
#: `kernmini` and no more, but imports `fastcore` and `microio`, and `kernmini` moved `debug` out
#: from under it after the version `ipymini` asks for, so resolving the declared requirements alone
#: installs a kernel that cannot start. These are the distributions, and `as_installed` supplies the
#: versions.
KERNEL_PACKAGES = ('ipykernel', 'ipymini', 'kernmini', 'fastcore', 'microio')

`KERNEL_PACKAGES` names distributions and no versions. `as_installed` supplies the versions, taken
from this process. `INSPECTOR` is left unpinned, so the two installers pin differently: the kernel
to whatever the host runs, the inspector to whatever the project resolves.

In [11]:
HOST_PY, INSPECTOR, KERNEL_PACKAGES

((3, 11),
 ('dhrishti',),
 ('ipykernel', 'ipymini', 'kernmini', 'fastcore', 'microio'))

In [12]:
#| export
_INSPECT = {}

In [13]:
#| export
_locks, _locks_lock = {}, threading.Lock()

In [14]:
#| export
from kunda.pythons import BUNDLE_ONLY, clean_env, strip_bundle

In [15]:
#| export
def work_dir(cwd=None):
    """`cwd` as a string, or a directory a child can actually be started in.

    A double-clicked app inherits `/` as its working directory, and one started from `open` inherits
    the bundle, which is read-only and code-signed. Either way a kernel launched there cannot write
    a file beside itself, and the error it gives says nothing about why.
    """
    if cwd: return str(cwd)
    here = os.getcwd()
    if not getattr(sys, 'frozen', False): return here
    return here if here != os.sep and os.access(here, os.W_OK) else str(Path.home())

A truthy `cwd` comes back as a string and is never examined, so a caller that names a directory
keeps it. `None` gives this process's working directory. In a frozen build `/` and a directory this
process cannot write to both become the home directory instead.

In [16]:
work_dir(Path('/repo/notebooks'))

'/repo/notebooks'

In [17]:
#| hide
here = os.getcwd()
try:
    sys.frozen = True; os.chdir(os.sep)
    test_eq(work_dir(), str(Path.home()))   # `/` is what a double-clicked app inherits
    test_eq(work_dir('/repo'), '/repo')     # a named directory is still kept
finally: os.chdir(here); del sys.frozen
test_eq(work_dir(), here)

In [18]:
#| export
def support_paths():
    "The `sys.path` entries `bootstrap_src` hands a kernel to borrow from."
    return [p for p in sys.path if p and os.path.isabs(p) and os.path.exists(p)]

Every relative entry is dropped, the empty one included, and so is every entry that is gone. A
relative entry means the working directory of whatever reads it, and a kernel's is not this one.

In [19]:
#| export
def _lock_for(python):
    "One lock per interpreter: two installers in one venv is a race, and three surfaces offer it."
    with _locks_lock: return _locks.setdefault(str(python), threading.Lock())

The table is keyed by the interpreter path as a string, so two callers spelling one interpreter
differently take different locks. Nothing is ever removed from the table.

In [20]:
test_is(_lock_for('/a/.venv/bin/python'), _lock_for('/a/.venv/bin/python'))
assert _lock_for('/a/.venv/bin/python') is not _lock_for('/b/.venv/bin/python')

In [21]:
#| export
class _Failed:
    "What a command that never started looks like, so a probe reports instead of raising."
    returncode, stdout = 127, ''
    def __init__(self, err): self.stderr = err

def _run(args, timeout=180):
    """Run a command and hand back its result, whether or not it ran.

    An interpreter that is not there is exactly what `kernel_support` exists to report, so the
    `FileNotFoundError` from spawning it is an answer rather than an error. A timeout is the same
    kind of answer: the environment is unusable, and saying which is the caller's business.
    """
    try:
        return subprocess.run([str(x) for x in args], env=clean_env(), text=True,
            encoding='utf-8', errors='replace', capture_output=True, timeout=timeout)
    except subprocess.TimeoutExpired: return _Failed(f'timed out after {timeout}s')
    except OSError as e: return _Failed(str(e))

`_run` answers with `returncode`, `stdout` and `stderr` whatever happens. An interpreter that is
not there and a command that runs past `timeout` both come back as `_Failed`: return code 127,
empty `stdout`, and the reason in `stderr`. The child is given `clean_env()`, this process's
environment with a frozen host's interpreter redirection removed.

In [22]:
p = _run(['/definitely/not/a/python', '-c', 'pass'])
p.returncode, p.stderr

(127, "[Errno 2] No such file or directory: '/definitely/not/a/python'")

In [23]:
#| hide
p = _run([sys.executable, '-c', 'import time; time.sleep(5)'], timeout=1)
test_eq((p.returncode, p.stdout), (127, ''))
test_eq(p.stderr, 'timed out after 1s')

In [24]:
#| export
def _last_line(text):
    "The line a traceback ends on. numpy's ABI failure wraps its own in eighteen lines of advice."
    return next((l for l in reversed((text or '').strip().splitlines()) if l.strip()), '').strip()

Empty text and `None` both give `''`.

In [25]:
_last_line('Traceback (most recent call last):\n  File "<string>", line 1\nImportError: numpy ABI\n\n')

'ImportError: numpy ABI'

In [26]:
#| hide
test_eq(_last_line(''), '')
test_eq(_last_line(None), '')
test_eq(_last_line('  only line  '), 'only line')

In [27]:
#| export
def _probe(python, module, paths=()):
    "Whether `module` imports in `python`, with its version or the failure."
    # The same question the bootstrap asks, and asked the same way: paths only where the version
    # matches. Extending regardless is what made a 3.13 kernel fail on the bundle's 3.12 bytecode.
    pre = (f'import sys\nif tuple(sys.version_info[:2]) == {HOST_PY!r}: '
           f'sys.path.extend({list(paths)!r})\n') if paths else ''
    p = _run([python, '-c', f'{pre}import {module}; print(getattr({module}, "__version__", "installed"))'], 15)
    return {'available': p.returncode == 0,
        'version': p.stdout.strip() if p.returncode == 0 else '',
        'error': _last_line(p.stderr or p.stdout) if p.returncode else ''}

Three keys, always: `available`, `version` and `error`. A module that has no `__version__` reports
`'installed'`. `error` is the last line of the failure, and `''` where there was none.

`paths` is added to the child's `sys.path` only where the child's minor version is `HOST_PY`. The
entries are written into the child's source as a literal, so they have to be strings.

In [28]:
tmp = TemporaryDirectory(); d = Path(tmp.name)
(d/'borrowed.py').write_text('__version__ = "1.2"')
_probe(sys.executable, 'borrowed', paths=[str(d)])

{'available': True, 'version': '1.2', 'error': ''}

In [29]:
#| hide
miss = _probe(sys.executable, 'borrowed')
test_eq((miss['available'], miss['version']), (False, ''))
assert 'ModuleNotFoundError' in miss['error']
test_eq(_probe(sys.executable, 'os')['version'], 'installed')

In [30]:
#| export
def kernel_support(python):
    "Importability of both supported kernel launchers in ``python``."
    python = str(Path(python).expanduser())
    return {m: _probe(python, m) for m in ('ipykernel', 'ipymini')}

Both launchers are asked every time, so a caller can tell which of them the environment has. `~` in
`python` is expanded. An interpreter that does not exist is reported, not raised about: the error
takes the place of the version.

This process's `sys.path` is not offered. A launcher has to be installed in the environment it runs
in.

In [31]:
ok = kernel_support(sys.executable)['ipymini']
gone = kernel_support('/definitely/not/a/python')['ipymini']
ok, gone

({'available': True, 'version': '0.1.21', 'error': ''},
 {'available': False,
  'version': '',
  'error': "[Errno 2] No such file or directory: '/definitely/not/a/python'"})

In [32]:
#| export
def inspector_support(python, refresh=False):
    """Whether a kernel on `python` could import the live-variable inspector.

    Asked of the pinned dhrishti itself rather than of a list of package names, so a dependency
    added upstream is answered for too, and so the answer includes what stopped it.
    """
    python = str(Path(python).expanduser())
    hit = _INSPECT.get(python)
    if not refresh and hit and time.time() - hit[0] < INSPECTOR_TTL: return hit[1]
    state = _probe(python, 'dhrishti.serving', support_paths())
    _INSPECT[python] = (time.time(), state)
    return state

One answer per interpreter is kept for `INSPECTOR_TTL` seconds and handed back as the same object.
`refresh=True` probes again and replaces it. The cache is process-wide and is never emptied, so an
interpreter that has just been changed is believed once the entry expires or a caller asks for a
refresh.

The probe lends the child this process's `sys.path`, so a kernel on the host's minor version needs
no dhrishti of its own. On any other version the paths are ignored and the answer is about that
environment alone.

In [33]:
inspector_support(sys.executable)

{'available': False,
 'version': '',
 'error': "ModuleNotFoundError: No module named 'dhrishti'"}

In [34]:
#| hide
s = inspector_support(sys.executable)
test_is(inspector_support(sys.executable), s)
_INSPECT[sys.executable] = (time.time() - INSPECTOR_TTL - 1, {'available': 'stale'})
test_eq(set(inspector_support(sys.executable)), {'available', 'version', 'error'})   # expired, asked again

In [35]:
#| export
def installable(python):
    "Whether anything may install into `python`. Never this one, and never inside a signed bundle."
    if not python: return False
    p = os.path.abspath(str(python))
    if p == os.path.abspath(sys.executable): return False
    return '/Contents/Resources/' not in p and '/Contents/MacOS/' not in p

Three refusals: nothing, this interpreter, and anything under a macOS bundle's `Contents/Resources`
or `Contents/MacOS`. Everything else is allowed, an interpreter that does not exist included.

This interpreter is recognised by its absolute path, with symlinks left as they are. A second name
for the same interpreter passes.

In [36]:
installable(sys.executable), installable('/repo/.venv/bin/python')

(False, True)

In [37]:
#| export
def as_installed(names=KERNEL_PACKAGES):
    """`names` pinned to the versions this process is running, and left loose where it has none.

    The host's own environment is the one combination of these known to start a kernel: it was
    resolved from a lock and tested. Asking an index for the newest of each instead is what
    produced `ipymini 0.1.20` beside `kernmini 0.1.9`, which import each other and do not fit.
    When the host upgrades, so does what it installs.
    """
    from importlib.metadata import PackageNotFoundError, version
    out = []
    for name in names:
        try: out.append(f'{name}=={version(name)}')
        except (PackageNotFoundError, ValueError, OSError): out.append(name)
    return out

The names are distribution names, read from this process's installed metadata rather than from
imports. A distribution the host does not have goes through unchanged and the resolver picks the
version.

In [38]:
as_installed(['fastcore', 'ipymini', 'not_a_real_package'])

['fastcore==2.2.19', 'ipymini==0.1.21', 'not_a_real_package']

In [39]:
#| export
def _attempts(python, packages):
    "Installer commands to try, in order, for environments that intentionally lack pip."
    out = []
    if uv := shutil.which('uv'): out.append([uv, 'pip', 'install', '--python', python, *packages])
    out.append([python, '-m', 'pip', 'install', *packages])
    if not uv and _run([python, '-m', 'ensurepip', '--upgrade']).returncode == 0:
        out.insert(0, [python, '-m', 'pip', 'install', *packages])
    return out

The last attempt is always `python -m pip install`. Where `uv` is on `PATH` it goes first, and it
installs into `python` without needing anything inside it.

Building the list runs a command. Where `uv` is missing, `_attempts` runs
`python -m ensurepip --upgrade` and, when that succeeds, puts a pip attempt at the front, so an
environment that ships without pip has one before the first attempt is made. The pip command is
then listed twice.

In [40]:
_attempts('/nowhere/bin/python', ['ipymini==0.1.21'])[-1]

['/nowhere/bin/python', '-m', 'pip', 'install', 'ipymini==0.1.21']

In [41]:
#| hide
cmds = _attempts('/nowhere/bin/python', ['a', 'b'])
test_eq(cmds[-1], ['/nowhere/bin/python', '-m', 'pip', 'install', 'a', 'b'])
if shutil.which('uv'): test_eq(cmds[0][1:5], ['pip', 'install', '--python', '/nowhere/bin/python'])

In [42]:
#| export
def _log(command, p):
    return {'command': command, 'returncode': p.returncode,
        'output': (p.stdout + '\n' + p.stderr).strip()[-8000:]}

In [43]:
#| export
def _detail(logs):
    return next((a['output'] for a in reversed(logs) if a['output']), 'no installer answered')

`_log` keeps the command, its return code, and the last 8000 characters of `stdout` and `stderr`
joined. A resolver that fails prints far more than that, and the end of it is the part that says
why.

`_detail` is what a raised message ends with: the output of the last attempt that said anything, or
`no installer answered` where none did.

In [44]:
#| hide
logs = [_log(['pip'], _Failed('')), _log(['uv'], _Failed('no interpreter'))]
test_eq(logs[0], {'command': ['pip'], 'returncode': 127, 'output': ''})
test_eq(_detail(logs), 'no interpreter')
test_eq(_detail(logs[:1]), 'no installer answered')
test_eq(_log(['pip'], _Failed('x' * 9000))['output'], 'x' * 8000)

In [45]:
#| export
def install_kernel_support(python, packages=None):
    """Install kernel packages, preferring uv for environments intentionally lacking pip.

    An installer that returned 0 and left a kernel that will not import is a failure of the
    packages, not of the installer, and saying so is the difference between a report about
    `ipymini` and one about the `pip` a uv environment was never going to have.
    """
    python = str(Path(python).expanduser().absolute())
    packages, logs, unimportable = list(packages or as_installed()), [], ''
    with _lock_for(python):
        for command in _attempts(python, packages):
            p = _run(command, 600)
            logs.append(_log(command, p))
            if p.returncode == 0:
                support = kernel_support(python)
                if all(x['available'] for x in support.values()):
                    return {'ok': True, 'python': python, 'support': support, 'attempts': logs}
                unimportable = '; '.join(f"{name} installed but does not import ({x['error']})"
                                         for name, x in support.items() if not x['available'])
    raise RuntimeError(f'kernel support installation failed: {unimportable or _detail(logs)}')

`install_kernel_support` returns only where the install succeeded and both launchers then import.
Anything else raises `RuntimeError`. There is no failure return value. The message names whichever
launcher installed but does not import, or gives the output of the last installer that said
anything.

An installer that returns 0 and leaves a kernel that will not import does not end the run. The next
attempt in the list is still made.

`installable` is not consulted. Whether this environment may be written to is the caller's
decision.

In [46]:
#| hide
# No installer can start against a path that is not an interpreter, so this installs nothing.
test_fail(install_kernel_support, args=['/nowhere/bin/python'], contains='kernel support installation failed')

In [47]:
#| export
def install_inspector_support(python, packages=INSPECTOR):
    "Give `python` its own copy of what the inspector imports, then ask it again."
    python = str(Path(python).expanduser().absolute())
    logs = []
    with _lock_for(python):
        for command in _attempts(python, packages):
            p = _run(command, 600)
            logs.append(_log(command, p))
            if p.returncode != 0: continue
            state = inspector_support(python, refresh=True)
            if state['available']:
                return {'ok': True, 'python': python, 'inspector': state, 'attempts': logs}
            raise RuntimeError(f'{" and ".join(packages)} are installed, and the inspector still '
                               f'will not import: {state["error"]}')
    raise RuntimeError(f'could not install {" and ".join(packages)}: {_detail(logs)}')

`install_inspector_support` installs `INSPECTOR`, then asks `inspector_support` again with
`refresh=True`, so a cached answer cannot make a fresh install look like a failure.

The first installer that returns 0 settles it. Where the inspector still does not import,
`install_inspector_support` raises there instead of trying the next installer, which is where it
differs from `install_kernel_support`.

In [48]:
#| hide
tmp.cleanup()